# ETo results overview

This notebook is the primary narrative notebook for the repository. It does not recalculate ETo methods or duplicate metric formulas; it reads the tables, reports, and figures generated by the pipeline.

Run the pipeline first from the repository root:

```bash
python -m scripts.cli all --year 2024
python -m scripts.cli validate-data --year 2024
```

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from scripts.config import OUTPUTS_FIGURES, OUTPUTS_REPORTS, OUTPUTS_TABLES, SITES

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)

In [ ]:
required_paths = [OUTPUTS_TABLES, OUTPUTS_FIGURES, OUTPUTS_REPORTS]
missing = [path for path in required_paths if not Path(path).exists()]
if missing:
    raise FileNotFoundError(
        "Missing generated outputs. Run `python -m scripts.cli all --year 2024` "
        "and `python -m scripts.cli validate-data --year 2024` first. Missing: "
        + ", ".join(str(path) for path in missing)
    )

list(SITES.keys())

## Daily performance metrics

The tables below are generated by `scripts.metrics.compute_metrics` through the CLI. Lower RMSE/MAE indicates smaller errors against FAO-56 Penman-Monteith; MBE indicates signed bias; higher R2 and Willmott's d indicate stronger agreement.

In [ ]:
daily_metrics = {
    site: pd.read_csv(OUTPUTS_TABLES / f"{site}_daily_metrics.csv")
    for site in SITES
}

for site, table in daily_metrics.items():
    print(f"\n{site.upper()} daily metrics")
    display(table.sort_values("rmse"))

## Monthly performance metrics

Monthly metrics use the same metric functions after monthly aggregation by the pipeline.

In [ ]:
monthly_metrics = {
    site: pd.read_csv(OUTPUTS_TABLES / f"{site}_monthly_metrics.csv")
    for site in SITES
}

for site, table in monthly_metrics.items():
    print(f"\n{site.upper()} monthly metrics")
    display(table.sort_values("rmse"))

## Data quality audit

These reports are generated by `python -m scripts.cli validate-data --year 2024`. They document missing dates, duplicate dates, missing values, interpolated values, and conservative physical-limit flags.

In [ ]:
quality_reports = {
    site: pd.read_csv(OUTPUTS_REPORTS / f"{site}_data_quality.csv")
    for site in SITES
}

for site, report in quality_reports.items():
    print(f"\n{site.upper()} variables with audit flags")
    flagged = report.loc[
        (report["missing_values"] > 0)
        | (report["interpolated_values"] > 0)
        | (report["physical_limit_violations"] > 0)
        | report["missing_dates"].fillna("").ne("")
        | report["duplicate_dates"].fillna("").ne("")
    ]
    display(flagged if not flagged.empty else report.head(0))

## Generated figures

The figures below are loaded from `outputs/figures/`; the notebook does not regenerate plots.

In [ ]:
for site in SITES:
    print(f"\n{site.upper()} monthly totals")
    display(Image(filename=str(OUTPUTS_FIGURES / site / f"{site}_monthly_totals.png")))
    print(f"{site.upper()} daily Taylor diagram")
    display(Image(filename=str(OUTPUTS_FIGURES / site / f"{site}_daily_taylor.png")))